In [1]:
import pandas as pd
import numpy as np
import os
import glob
from IPython.display import display

# Configurações
pd.set_option('display.max_columns', None)
print("Bibliotecas importadas.")

Bibliotecas importadas.


In [2]:
# O caminho base é o diretório atual
base_path = '.' 

# Lista de pastas de ataque
attack_folders = [
    'analysis',
    'dos',
    'exploits',
    'fuzzers',
    'reconnaissance'
]

# Lista de todos os diretórios a serem processados
data_paths = {
    'Normal': os.path.join(base_path, 'normal')
}

# Adiciona os caminhos de ataque ao dicionário
for attack in attack_folders:
    data_paths[attack.capitalize()] = os.path.join(base_path, 'ataque', attack)

print("Caminhos de dados definidos:")
print(data_paths)

# Lista para armazenar todos os DataFrames
all_dataframes = []

# Loop para encontrar e ler todos os arquivos '*_features_processed.csv'
print("\nIniciando leitura dos arquivos '_features_processed.csv'...")

for category_name, folder_path in data_paths.items():
    if not os.path.isdir(folder_path):
        print(f"Aviso: Diretório não encontrado, pulando: {folder_path}")
        continue
    
    # Usa glob para encontrar o arquivo '*_features_processed.csv' dentro da pasta
    search_pattern = os.path.join(folder_path, '*_features_processed.csv')
    found_files = glob.glob(search_pattern)
    
    if not found_files:
        print(f"Aviso: Nenhum arquivo '_features_processed.csv' encontrado em: {folder_path}")
        continue
    
    # Assume que há apenas um arquivo _features_processed.csv por pasta
    file_path = found_files[0]
    
    try:
        print(f"Lendo: {file_path} ...")
        df_temp = pd.read_csv(file_path, low_memory=False)
        all_dataframes.append(df_temp)
    except Exception as e:
        print(f"Erro ao ler {file_path}: {e}")

print("\nLeitura de todos os arquivos concluída.")

Caminhos de dados definidos:
{'Normal': '.\\normal', 'Analysis': '.\\ataque\\analysis', 'Dos': '.\\ataque\\dos', 'Exploits': '.\\ataque\\exploits', 'Fuzzers': '.\\ataque\\fuzzers', 'Reconnaissance': '.\\ataque\\reconnaissance'}

Iniciando leitura dos arquivos '_features_processed.csv'...
Lendo: .\normal\normal_features_processed.csv ...
Lendo: .\ataque\analysis\analysis_features_processed.csv ...
Lendo: .\ataque\dos\dos_features_processed.csv ...
Lendo: .\ataque\exploits\exploits_features_processed.csv ...
Lendo: .\ataque\fuzzers\fuzzers_features_processed.csv ...
Lendo: .\ataque\reconnaissance\reconnaissance_features_processed.csv ...

Leitura de todos os arquivos concluída.


In [3]:
if not all_dataframes:
    print("ERRO: Nenhum DataFrame foi carregado. Não é possível continuar.")
else:
    # Juntar todos os DataFrames em um único dataset
    print("Concatenando todos os DataFrames...")
    final_dataset = pd.concat(all_dataframes, ignore_index=True)

    print("\nIniciando limpeza de valores NaN...")
    print(f"Total de valores nulos (NaN) antes da limpeza: {final_dataset.isnull().sum().sum()}")

    # Definir colunas com base no NUSW-NB15_features.csv
    # Tipos nominais (preencher com '-')
    # Inclui: nominal, Binary, binary
    nominal_cols = [
        'srcip', 'dstip', 'proto', 'state', 'service', 
        'is_sm_ips_ports', 'is_ftp_login', 
        'attack_cat', 'Label'
    ]
    
    # Tipos numéricos (preencher com 0), conforme arquivo-guia
    # Inclui: integer, Float, Timestamp
    numeric_cols = [
        'sport', 'dsport', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 
        'sloss', 'dloss', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 
        'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 
        'response_body_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 
        'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'ct_state_ttl', 
        'ct_flw_http_mthd', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 
        'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 
        'ct_dst_sport_ltm', 'ct_dst_src_ltm'
    ]

    # Identificar quais dessas colunas realmente existem no nosso DataFrame
    # (Pode ser que alguma coluna não tenha sido capturada ou gerada)
    nominal_cols_to_fill = [col for col in nominal_cols if col in final_dataset.columns]
    numeric_cols_to_fill = [col for col in numeric_cols if col in final_dataset.columns]
    
    # Substituir valores infinitos (inf) por 0 (APENAS em colunas numéricas)
    # Fazemos isso primeiro para garantir que 'inf' não seja tratado como string.
    if numeric_cols_to_fill:
         final_dataset[numeric_cols_to_fill] = final_dataset[numeric_cols_to_fill].replace([np.inf, -np.inf], 0)
    print("Valores infinitos (inf) em colunas numéricas substituídos por 0.")

    # Padronizar todos os valores "null-like" (incluindo strings vazias) para np.nan
    # Isso força strings vazias (''), 'None', 'NULL', etc., a se tornarem np.nan
    # para que o fillna() possa pegá-los.
    null_like_strings = ['', ' ', 'None', 'NULL', 'N/A', 'nan', 'NaN', 'undefined', '-']
    final_dataset.replace(null_like_strings, np.nan, inplace=True)
    print(f"Strings 'null-like' (incluindo '') convertidas para np.nan.")
    print(f"Total de NaNs para preencher agora: {final_dataset.isnull().sum().sum()}")
    
    # Aplicar preenchimento de NaN (agora que TODOS os nulos são np.nan)
    if nominal_cols_to_fill:
        final_dataset[nominal_cols_to_fill] = final_dataset[nominal_cols_to_fill].fillna('-')
        print(f"Valores NaN em {len(nominal_cols_to_fill)} colunas nominais/binárias preenchidos com '-'.")

    if numeric_cols_to_fill:
        final_dataset[numeric_cols_to_fill] = final_dataset[numeric_cols_to_fill].fillna(0)
        print(f"Valores NaN em {len(numeric_cols_to_fill)} colunas numéricas preenchidos com 0.")

    # Verificação final de NaNs restantes
    remaining_nans = final_dataset.isnull().sum().sum()
    print(f"Total de valores nulos (NaN) após a limpeza final: {remaining_nans}")
    
    if remaining_nans > 0:
        print("\nATENÇÃO: Ainda existem NaNs no DataFrame! Colunas com NaNs:")
        print(final_dataset.isnull().sum()[final_dataset.isnull().sum() > 0])

    
    # Balanceamento de amostras: Reduzir classe majoritária "Exploits"

    print("\nBalanceando a classe 'Exploits' para reduzir desbalanceamento...")

    # Contar instâncias por classe
    class_counts = final_dataset['attack_cat'].value_counts()
    print("Contagem original de classes:")
    print(class_counts)

    # Escolher o número alvo para Exploits (ex.: 10% do total original)
    target_fraction = 0.010  # Mantenha apenas 1% da classe Exploits
    target_fraction_dos = 0.10
    recon_total = class_counts.get('Exploits', 0)
    dos_total = class_counts.get('DoS', 0)
    target_size = int(recon_total * target_fraction)
    target_size_dos = int(recon_total * target_fraction)

    if recon_total > 0:
        # Filtrar Exploits
        df_recon = final_dataset[final_dataset['attack_cat'] == 'Exploits']
        df_other = final_dataset[final_dataset['attack_cat'] != 'Exploits']

        print(f"Reduzindo Exploits de {recon_total} → {target_size} amostras...")

        # Fazer amostragem estratificada aleatória (random_state para reprodutibilidade)
        df_recon_sampled = df_recon.sample(n=target_size, random_state=42)

        # Reunir o dataset balanceado
        final_dataset = pd.concat([df_other, df_recon_sampled], ignore_index=True)

        print("Distribuição após balanceamento:")
        print(final_dataset['attack_cat'].value_counts())
        
    if dos_total > 0:
        df_dos = final_dataset[final_dataset['attack_cat'] == 'DoS']
        df_other_dos = final_dataset[final_dataset['attack_cat'] != 'DoS']

        print(f"Reduzindo DoS de {dos_total} → {target_size_dos} amostras...")

        # Fazer amostragem estratificada aleatória (random_state para reprodutibilidade)
        df_dos_sampled = df_dos.sample(n=target_size_dos, random_state=42)

        # Reunir o dataset balanceado
        final_dataset = pd.concat([df_other_dos, df_dos_sampled], ignore_index=True)

        print("Distribuição após balanceamento:")
        print(final_dataset['attack_cat'].value_counts())
    else:
        print("Nenhuma amostra de DoS encontrada — nada a balancear.")

    
    # Embaralhar o dataset final
    # Isso quebra qualquer ordem cronológica da captura
    print("\nEmbaralhando o dataset final...")
    final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    print("##################################")
    print(final_dataset.isnull().values.any())

    # Salvar o dataset consolidado
    output_filename = 'ubuntu_server_dataset_final_argus.csv'
    print("#############################################################################")
    print(final_dataset.isnull().values.any())
    print("#############################################################################")
    final_dataset.to_csv(output_filename, index=False)
    print(f"\nDataset consolidado e limpo salvo com sucesso como: {output_filename}")
    # 4. Verificação Final
    print("\n--- Informações do Dataset Final Consolidado ---")
    final_dataset.info()
    
    print("\n--- Amostra do Dataset Final (5 primeiras linhas) ---")
    display(final_dataset.head())
    
    print("\n--- Distribuição de Classes no Dataset Final ---")
    # Mostra a contagem e a porcentagem de cada categoria
    print(final_dataset['attack_cat'].value_counts())
    print("\nDistribuição Percentual:")
    print(final_dataset['attack_cat'].value_counts(normalize=True) * 100)

Concatenando todos os DataFrames...

Iniciando limpeza de valores NaN...
Total de valores nulos (NaN) antes da limpeza: 141995
Valores infinitos (inf) em colunas numéricas substituídos por 0.
Strings 'null-like' (incluindo '') convertidas para np.nan.
Total de NaNs para preencher agora: 141995
Valores NaN em 6 colunas nominais/binárias preenchidos com '-'.
Valores NaN em 25 colunas numéricas preenchidos com 0.
Total de valores nulos (NaN) após a limpeza final: 0

Balanceando a classe 'Exploits' para reduzir desbalanceamento...
Contagem original de classes:
attack_cat
Exploits          278010
DoS                44069
Reconnaissance     17104
Fuzzers            16863
Normal              7689
Analysis             116
Name: count, dtype: int64
44069 BBBBBBBBBBBBBBBBBBBBB
Reduzindo Exploits de 278010 → 2780 amostras...
Distribuição após balanceamento:
attack_cat
DoS               44069
Reconnaissance    17104
Fuzzers           16863
Normal             7689
Exploits           2780
Analysis  

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,5.510268,tcp,-,SH,111.0,0.0,7677.0,0.0,20.144211,0,0,11145.737376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,69,0,0.0,0.0,50,100,100,100,2,100,0,0,50,100,50,0,Fuzzers,1
1,5.449793,tcp,-,SH,102.0,0.0,7126.0,0.0,18.716307,0,0,10460.580796,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,69,0,0.0,0.0,49,100,100,100,2,100,0,0,51,100,49,0,Fuzzers,1
2,0.053959,udp,-,INT,2.0,0.0,166.0,0.0,37.065179,40,0,24611.278934,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,83,0,0.0,0.0,60,68,100,1,14,100,0,0,30,100,60,0,Reconnaissance,1
3,2.294511,tcp,ftp,SH,19.0,1.0,99.0,0.0,8.716454,0,0,345.171586,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,0,0.0,0.0,18,100,94,18,1,94,1,18,35,95,18,0,Normal,0
4,5.623328,tcp,-,SH,104.0,0.0,6855.0,0.0,18.494386,0,0,9752.232130,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,65,0,0.0,0.0,49,100,99,99,2,99,0,0,50,99,49,0,Fuzzers,1



--- Distribuição de Classes no Dataset Final ---
attack_cat
Reconnaissance    17104
Fuzzers           16863
Normal             7689
DoS                2780
Exploits           2780
Analysis            116
Name: count, dtype: int64

Distribuição Percentual:
attack_cat
Reconnaissance    36.136229
Fuzzers           35.627060
Normal            16.244824
DoS                5.873405
Exploits           5.873405
Analysis           0.245077
Name: proportion, dtype: float64
